<a href="https://colab.research.google.com/github/cwf2/dices-examples/blob/main/colab/NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DICES + NLP: Using ancillary tools to download and parse speech text

The DICES database stores records of who speaks to whom, but not the text of speeches. This notebook demonstrate how to use additional resources to download the full text of speeches from Perseus, and to generate lemma and part-of-speech tags using spaCy.

Note that parsing depends not only on NLP software (spacy) but also on third-party language models.

### Example: Speeches in Silius Italicus

Parsing with language models goes slowly and uses a lot of memory. In this example we've chosen a set of texts that is large enough to give a sense of what can be done without taking an enormous amount of time (we tested on Google Colab's free tier). If it still takes too long, try making the set of speeches smaller. If you want to work with Greek texts, adjust where noted below.

## Preliminaries
### Install models and spaCy
 - **You must restart kernel afterwards**
 - Don't re-run this cell after restart — just carry on below

In [1]:
# Latin transformers model
%pip install https://huggingface.co/latincy/la_core_web_trf/resolve/main/la_core_web_trf-3.9.5-py3-none-any.whl

# Greek transformers model - uncomment the line below if you're parsing Greek
# %pip install https://huggingface.co/chcaa/grc_odycy_joint_trf/resolve/main/grc_odycy_joint_trf-0.7.0-py3-none-any.whl

# DICES client with NLP prereqs
%pip install "dices-client[spacy] @ git+https://github.com/cwf2/dices-client.git"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 GB ? eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.7/158.7 kB 10.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 795.8/795.8 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.4/313.4 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 83.4 MB/s eta 0:00:00
  Created wheel for latincy-preprocess: filename=latincy_preprocess-0.2.0-cp312-cp312-linux_x86_64.whl size=482078

## Main code begins here

### Import statements

In [2]:
# DICES client
from dicesapi import DicesAPI

# Pandas for tabular data
import pandas as pd

# tqdm is a simple progress bar for long tasks
from tqdm import tqdm

### Initialize new DICES session

If something hasn't been installed correctly, these may throw errors. Warnings about spacy version can be ignored. If you install any new packages, restart the kernel and begin again from the import statements above.

In [ ]:
api = DicesAPI()

# this step enables the api to download texts from Perseus
api.initializeCts()

# this step enables NLP with spacy
api.initializeNlp(
    latin_model="la_core_web_trf",
#    greek_model="grc_odycy_joint_trf",  # uncomment if Greek model is installed
)

### Query DICES for Silius' speeches

In [ ]:
# Download speech records
speeches = api.getSpeeches(author_name="Silius")

print(f"got {len(speeches)} speeches")

### Download text passages from Perseus

To download the text of a speech from Perseus, use `fetchPassage()`. Perseus has most, but not all, of the texts in DICES. If a passage is retrieved, it will populate the speech's `passage` attribute. If the text is not in Perseus, `passage` will be `None`.

💁🏻‍♂️ If the Perseus server stops responding, try running the loop again. Successfully retrieved speeches are cached, so it will pick up where it left off.

**Note to long-time users**: This process has changed since our beta version. Use `fetchPassage()` rather than the old `CtsAPI.getCTS()`.

In [ ]:
for s in tqdm(speeches):
    s.fetchPassage()

### Parse with spaCy

The DICES client provides a convenience method that wraps spaCy's NLP pipeline. Once a speech has been downloaded and `passage` has been created, you will be able to run the `passage.runSpacyPipeline()`.

In [ ]:
for s in tqdm(speeches):
    if s.passage is not None:
        s.passage.runSpacyPipeline()

## Build token table

The spaCy pipeline creates a new attribute on `passage`: `passage.spacy_doc`. This contains, among other things, an iterable collection of tokens (words and punctuation marks). While spaCy provides its own methods for working with this data in Python, we find it easiest to turn the tagged tokens into a table with Pandas. This can then be exported in CSV format if you prefer to do your analysis in Excel.

### spaCy token attributes

The main features of spaCy tokens that we work with here are:
 - `text`, a string representing the inflected form as it appears in the text
 - `lemma_`, a string representing the dictionary form
 - `pos_`, a string representing the part of speech tag
 - `morph`, a bundle of morphological attributes that works like a dictionary: see the code below for specific fields pulled with `.get()`.

💁🏻‍♂️ spaCy part of speech and morphological categories are taken from the [Universal Dependencies Framework](https://universaldependencies.org/u/feat/index.html)

### DICES helper methods

spaCy's NLP pipeline treats the text as a single string, but all our texts are hexameter verse. We've added a couple helper methods and attributes to trace tokens back to their locus:
 - `line_array` contains the Perseus text broken into a list of lines with their loci.
 - `getLineIndex(token)` returns an index into `line_array` to find the corresponding line for a given spaCy token.


In [ ]:
# build a table with one row per token
# start with an empty list of rows
tokens = []

# iterate over all speeches
for s in speeches:

    # skip any speeches that failed to download
    if s.passage is None:
        continue
    # skip any speeches that failed to parse
    if s.passage.spacy_doc is None:
        continue

    # iterate over all the tokens in each speech
    for token in s.passage.spacy_doc:

        # get the index for this token
        i = s.passage.getLineIndex(tok)

        # get prefix for locus (everything before the last ".")
        pref = s.l_fi.split(".")[0]

        # create a record for this token
        token = {
            # attributes of the speech
            "author": s.work.author.name,
            "work": s.work.title,
            "pref": pref, # i.e. book number
            "seq": s.passage.line_array[i]["seq"], # a sortable integer, since
            "line": s.passage.line_array[i]["n"], # line numbers can be out of order
            "spkr": s.getSpkrString(), # all speakers as a single string
            "addr": s.getAddrString(), # all addressees as a single string
            "type": s.type, # (soliloquy, monologue, dialogue, general)
            "tags": ";".join([tag["type"] for tag in s._attributes["tags"]]),
                            # e.g. prayer, lament, exhortation, etc.

            # attributes of this token
            token = tok.text,  # inflected form
            lemma = tok.lemma_, # dictionary headword
            pos = tok.pos_, # part of speech
            verbform = ";".join(tok.morph.get("VerbForm")), # eg. finite, infinitive, participle
            mood = ";".join(tok.morph.get("Mood")),
            tense = ";".join(tok.morph.get("Tense")),
            voice = ";".join(tok.morph.get("Voice")),
            person = ";".join(tok.morph.get("Person")),
            number = ";".join(tok.morph.get("Number")),
            case = ";".join(tok.morph.get("Case")),
            gender = ";".join(tok.morph.get("Gender")),
        }

        # add this record to the list
        tokens.append(token)

# convert to a dataframe
tokens = pd.DataFrame(tokens)

# show table
display(tokens)

### Export as Excel file

In [ ]:
from google.colab import files

# Convert DataFrame to Excel
tokens.to_excel("sil_tokens.xlsx", index=False)

# Provide download link
files.download("sil_tokens.xlsx")
